# Hyperedge Structural Entropy $S_e$

Implementation of the hyperedge structural entropy proposed by Xian et al. (2026),
using the [DeepHypergraph](https://github.com/iMoonLab/DeepHypergraph) library.

Unlike the spectral entropy from `entropy_vector_dhg.ipynb`, this measure weights
hyperedges by their **order** and the **hyperdegrees** of their constituent nodes,
capturing dual sources of structural uncertainty.

In [ ]:
import numpy as np
from collections import Counter
from dhg import Hypergraph

## 1. Core Implementation

Given a `dhg.Hypergraph` $H = (V, E)$:

1. **Incidence matrix** $A$ (shape $E_{\text{total}} \times |V|$) from `hg.H_T`
2. **Node hyperdegree** $D(i) = \sum_j a_{ji}$ — column sums of $A$
3. **Hyperedge order** $O(e) = \sum_j a_{ej}$ — row sums of $A$
4. **Hyperedge weight** $w(e) = O(e) \times \sum_{i \in n_e} D(i)$
5. **Probability** $P(w) = n_w / E_{\text{total}}$ where $n_w$ = number of edges with weight $w$
6. **Entropy** $S_e = -\sum_{w \in W} P(w) \ln P(w)$
7. **Normalised** $S_{\text{norm}} = S_e / \ln(E_{\text{total}})$

In [ ]:
def compute_hyperedge_structural_entropy(hg):
    """
    Compute hyperedge structural entropy S_e for a dhg.Hypergraph.

    Args:
        hg: dhg.Hypergraph instance.

    Returns:
        S_e:          float, raw Shannon entropy (nats).
        S_normalized: float, entropy normalised by ln(E_total).
        weights:      np.ndarray of per-hyperedge weights.
    """
    # A: (E_total, |V|) incidence matrix
    A = hg.H_T.to_dense().numpy().astype(np.float64)  # H_T shape is (num_e, num_v)
    E_total = hg.num_e

    # Step 1: Node hyperdegrees D(i) = column sums of A
    D = A.sum(axis=0)  # shape (|V|,)

    # Step 2: Hyperedge weights w(e) = O(e) * sum_{i in n_e} D(i)
    O = A.sum(axis=1)            # hyperedge orders, shape (E_total,)
    sum_D = A @ D                # sum of node hyperdegrees per edge, shape (E_total,)
    weights = O * sum_D          # shape (E_total,)

    # Step 3: Probability distribution P(w) = n_w / E_total
    weight_counts = Counter(weights.tolist())
    unique_weights = np.array(list(weight_counts.keys()))
    counts = np.array(list(weight_counts.values()), dtype=np.float64)
    P = counts / E_total

    # Step 4: Shannon entropy S_e = -sum P(w) ln P(w)
    S_e = -np.sum(P * np.log(P))

    # Normalization: S_max = ln(E_total)
    S_max = np.log(E_total) if E_total > 1 else 1.0
    S_normalized = S_e / S_max if S_max > 0 else 0.0

    return S_e, S_normalized, weights

In [ ]:
def compute_hyperedge_structural_entropy_batch(
    hypergraphs: list,
) -> np.ndarray:
    """
    Compute hyperedge structural entropy for a batch of dhg.Hypergraph objects.

    Args:
        hypergraphs: List of dhg.Hypergraph instances (length n_hypergraphs).

    Returns:
        entropies: np.ndarray of shape (n_hypergraphs, 2).
                   Column 0: S_e          — raw Shannon entropy (nats).
                   Column 1: S_normalized — entropy normalised by ln(E_total).
    """
    results = []
    for hg in hypergraphs:
        S_e, S_norm, _ = compute_hyperedge_structural_entropy(hg)
        results.append([S_e, S_norm])
    return np.array(results, dtype=np.float64)  # (n_hypergraphs, 2)


## 2. Worked Example

Step-by-step walkthrough on a small hypergraph to verify the calculation.

In [ ]:
hg = Hypergraph(6, [[0, 1, 2], [3, 4], [4, 5]])

A = hg.H_T.to_dense().numpy().astype(np.float64)
E_total = hg.num_e

print(f"Vertices: {hg.num_v},  Hyperedges: {hg.num_e}")
print(f"Edge list: {hg.e[0]}")
print(f"\nIncidence matrix A (E_total × |V|):")
print(A)

# Step 1
D = A.sum(axis=0)
print(f"\nStep 1 — Node hyperdegrees D(i): {D}")

# Step 2
O = A.sum(axis=1)
sum_D = A @ D
weights = O * sum_D
print(f"\nStep 2 — Hyperedge orders O(e):          {O}")
print(f"          Sum of node hyperdegrees Σ D(i): {sum_D}")
print(f"          Hyperedge weights w(e):           {weights}")

# Step 3
weight_counts = Counter(weights.tolist())
print(f"\nStep 3 — Weight counts: {dict(weight_counts)}")
P = np.array(list(weight_counts.values()), dtype=np.float64) / E_total
print(f"          Probabilities P(w): {P}")

# Step 4
S_e = -np.sum(P * np.log(P))
S_max = np.log(E_total)
S_norm = S_e / S_max
print(f"\nStep 4 — S_e          = {S_e:.6f} nats")
print(f"          S_max         = {S_max:.6f} nats")
print(f"          S_normalized  = {S_norm:.6f}")

## 3. Structural Sensitivity

Compare $S_e$ across hypergraphs with different structural properties on the same vertex set.

- **Uniform singleton** — all edges contain a single node: maximally disordered weights
- **Uniform full** — one edge containing all nodes: single weight → $S_e = 0$
- **ER-like** — random edges of varying sizes: high entropy
- **BA-like (star)** — hub-and-spoke structure: concentrated weights → low entropy

In [ ]:
cases = {
    "Singletons (max disorder)": Hypergraph(6, [[0], [1], [2], [3], [4], [5]]),
    "Single full edge (min)": Hypergraph(6, [[0, 1, 2, 3, 4, 5]]),
    "Paper example": Hypergraph(6, [[0, 1, 2], [3, 4], [4, 5]]),
    "Uniform 2-edges": Hypergraph(6, [[0, 1], [2, 3], [4, 5]]),
    "Star (hub=0)": Hypergraph(6, [[0, 1], [0, 2], [0, 3], [0, 4], [0, 5]]),
    "ER-like (mixed sizes)": Hypergraph(6, [[0, 1], [1, 2, 3], [2, 4], [3, 4, 5], [0, 5]]),
}

print(f"{'Hypergraph':<30} {'|V|':>4} {'|E|':>4} {'S_e':>10} {'S_norm':>10}  Weights")
print("-" * 85)
for name, hg in cases.items():
    S_e, S_norm, w = compute_hyperedge_structural_entropy(hg)
    print(f"{name:<30} {hg.num_v:>4} {hg.num_e:>4} {S_e:>10.4f} {S_norm:>10.4f}  {np.round(w, 1).tolist()}")

## 4. Scale-Invariance of Normalised Entropy

Verify that $S_{\text{norm}}$ remains comparable across hypergraphs of different sizes
by growing a uniform random hypergraph and a star hypergraph and tracking how entropy evolves.

In [ ]:
import random

random.seed(42)
np.random.seed(42)

sizes = [4, 8, 16, 32, 64]

print(f"{'|E|':>6}  {'ER S_norm':>12}  {'Star S_norm':>12}")
print("-" * 36)
for n_edges in sizes:
    n_vertices = n_edges * 2

    # ER-like: random edges of size 2 or 3
    er_edges = [
        sorted(random.sample(range(n_vertices), random.choice([2, 3])))
        for _ in range(n_edges)
    ]
    er_hg = Hypergraph(n_vertices, er_edges)
    _, er_norm, _ = compute_hyperedge_structural_entropy(er_hg)

    # Star: all edges share hub node 0
    star_edges = [[0, i + 1] for i in range(n_edges)]
    star_hg = Hypergraph(n_edges + 1, star_edges)
    _, star_norm, _ = compute_hyperedge_structural_entropy(star_hg)

    print(f"{n_edges:>6}  {er_norm:>12.4f}  {star_norm:>12.4f}")